# LeetCode 311 — Sparse Matrix Multiplication bằng Python

Link đề: https://leetcode.com/problems/sparse-matrix-multiplication/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Matrix

> Đây là bài **Premium**: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.

Skeleton LeetCode:

```python
class Solution:
    def multiply(self, mat1: List[List[int]], mat2: List[List[int]]) -> List[List[int]]:
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Nhớ lại **phép nhân ma trận** `C = A × B`: `C[i][j] = Σₖ A[i][k] · B[k][j]`, trong đó `A` cỡ `m × k`, `B` cỡ `k × n`, kết quả cỡ `m × n`.
2. Thấy vì sao bài này gọi là **ma trận thưa (sparse)**: phần lớn phần tử bằng `0`, nên có thể **bỏ qua các phép nhân vô ích** (`0 · x = 0`).
3. Đi từ **ba vòng lặp** → **bỏ qua `A[i][k] == 0` (đổi thứ tự vòng lặp `i, k, j`)** → **nén mỗi hàng của A thành danh sách `(k, giá trị)` khác 0** → **nén cả hai ma trận (dict / danh sách kề)** → **cách dùng `zip` trên ma trận chuyển vị (đối chiếu)**.
4. Biết cách **đếm số phép nhân** thực hiện để *đo* hiệu quả khi ma trận thưa.
5. Tránh các lỗi kinh điển: tạo kết quả bằng `[[0] * n] * m` (các hàng dùng chung bộ nhớ), dùng `=` thay vì `+=`, dùng `break` thay vì `continue` khi gặp số 0, và nhầm chiều của ma trận kết quả.

## 2. Đề bài & ví dụ

Cho hai ma trận thưa `mat1` cỡ `m × k` và `mat2` cỡ `k × n`. Trả về kết quả của `mat1 × mat2`. Có thể giả sử phép nhân luôn hợp lệ (số cột của `mat1` bằng số hàng của `mat2`).

**Ví dụ 1:**

```
mat1 = [[ 1, 0, 0],
        [-1, 0, 3]]

mat2 = [[ 7, 0, 0],
        [ 0, 0, 0],
        [ 0, 0, 1]]

Output: [[ 7, 0, 0],
         [-7, 0, 3]]
```

**Ví dụ 2:** `mat1 = [[0]]`, `mat2 = [[0]]` → `[[0]]`.

**Ràng buộc:** `m == mat1.length`; `k == mat1[i].length == mat2.length`; `n == mat2[i].length`; `1 <= m, n, k <= 100`; `-100 <= mat1[i][j], mat2[i][j] <= 100`.

In [1]:
from typing import List
import random

examples = [
    ([[1, 0, 0], [-1, 0, 3]], [[7, 0, 0], [0, 0, 0], [0, 0, 1]], [[7, 0, 0], [-7, 0, 3]]),
    ([[0]], [[0]], [[0]]),
]
for a, b, e in examples:
    print("mat1 =", a, "| mat2 =", b, "-> kỳ vọng", e)

mat1 = [[1, 0, 0], [-1, 0, 3]] | mat2 = [[7, 0, 0], [0, 0, 0], [0, 0, 1]] -> kỳ vọng [[7, 0, 0], [-7, 0, 3]]
mat1 = [[0]] | mat2 = [[0]] -> kỳ vọng [[0]]


## 3. Kiến thức nền

### 3.1. Nhân ma trận theo định nghĩa

`C[i][j]` là **tích vô hướng** của hàng `i` của `A` với cột `j` của `B`. Cần ba chỉ số: `i` chạy `m`, `j` chạy `n`, `k` chạy `k`: tổng `m · n · k` phép nhân.

In [2]:
A = [[1, 2], [3, 4]]
B = [[5, 6], [7, 8]]
C = [[sum(A[i][t] * B[t][j] for t in range(2)) for j in range(2)] for i in range(2)]
print(C)        # [[19, 22], [43, 50]]

[[19, 22], [43, 50]]


### 3.2. Ma trận thưa và "phép nhân vô ích"

Với `A[i][t] == 0` thì `A[i][t] · B[t][j] = 0` với **mọi** `j`: toàn bộ `n` phép nhân đó vô ích. Nếu ma trận có `90%` phần tử bằng `0`, ta có thể bỏ qua phần lớn công việc. Điều kiện bỏ qua nên đặt **sớm** (ở vòng lặp ngoài hơn) để tiết kiệm nhiều nhất.

In [3]:
A = [[1, 0, 0], [-1, 0, 3]]
zeros = sum(1 for row in A for x in row if x == 0)
print("số phần tử 0 của A:", zeros, "/", sum(len(r) for r in A))

số phần tử 0 của A: 3 / 6


### 3.3. Thứ tự vòng lặp `i, k, j` thân thiện với việc bỏ qua

Nếu đặt vòng `k` **ở giữa** (`i → k → j`), điều kiện `if A[i][k] == 0: continue` loại luôn cả vòng `j` bên trong. Còn với thứ tự `i → j → k` thì phải kiểm tra từng phần tử trong vòng trong cùng.

In [4]:
def count_ijk(A, B):
    m, k, n = len(A), len(B), len(B[0])
    return m * n * k            # thứ tự i, j, k: luôn đủ m*n*k phép nhân

def count_ikj_skip(A, B):
    m, k, n = len(A), len(B), len(B[0])
    return sum(n for i in range(m) for t in range(k) if A[i][t] != 0)   # chỉ tính khi A[i][t] khác 0

A = [[1, 0, 0], [-1, 0, 3]]
B = [[7, 0, 0], [0, 0, 0], [0, 0, 1]]
print("không bỏ qua:", count_ijk(A, B), "phép nhân | bỏ qua số 0 của A:", count_ikj_skip(A, B), "phép nhân")

không bỏ qua: 18 phép nhân | bỏ qua số 0 của A: 9 phép nhân


### 3.4. Biểu diễn thưa (sparse representation)

Mỗi hàng của ma trận thưa chỉ cần lưu các cặp `(cột, giá trị)` khác 0, ví dụ hàng `[-1, 0, 3]` → `[(0, -1), (2, 3)]`. Khi nhân, chỉ duyệt cặp có thật. Hàng của `B` cũng nén tương tự, nên cả hai ma trận đều thưa thì số phép nhân giảm mạnh.

In [5]:
def compress_rows(M):
    return [[(j, v) for j, v in enumerate(row) if v != 0] for row in M]

print(compress_rows([[-1, 0, 3], [0, 0, 0], [7, 0, 0]]))

[[(0, -1), (2, 3)], [], [(0, 7)]]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: `A = [[1,0,0],[-1,0,3]]`, `B = [[7,0,0],[0,0,0],[0,0,1]]`.

- Hàng 0 của `A`: chỉ `A[0][0] = 1` khác 0. Cộng `1 · (hàng 0 của B) = [7, 0, 0]`. Kết quả hàng 0: `[7, 0, 0]`.
- Hàng 1 của `A`: `A[1][0] = -1` và `A[1][2] = 3`. Cộng `−1 · [7,0,0] = [−7,0,0]` và `3 · (hàng 2 của B) = 3 · [0,0,1] = [0,0,3]`. Kết quả hàng 1: `[−7, 0, 3]`.

**Cách nhìn "hàng của C = tổ hợp tuyến tính các hàng của B"**: mỗi phần tử khác 0 `A[i][t]` kéo theo cộng `A[i][t] · B[t]` vào hàng `i` của kết quả. Hàng `t` của `B` bằng `0` thì cũng có thể bỏ qua luôn.

In [6]:
# Hàng của C = tổ hợp tuyến tính các hàng của B với hệ số lấy từ hàng của A
A = [[1, 0, 0], [-1, 0, 3]]
B = [[7, 0, 0], [0, 0, 0], [0, 0, 1]]
C = []
for row in A:
    acc = [0] * 3
    for t, coef in enumerate(row):
        if coef:
            acc = [x + coef * y for x, y in zip(acc, B[t])]
    C.append(acc)
print(C)

[[7, 0, 0], [-7, 0, 3]]


## 5. Cách 1 — Brute Force / baseline: ba vòng lặp theo định nghĩa

**Ý tưởng.** Với mọi `i, j` tính `Σₖ A[i][k] · B[k][j]`. Không bỏ qua gì cả.

In [7]:
def multiply_brute(mat1: List[List[int]], mat2: List[List[int]]) -> List[List[int]]:
    m, k, n = len(mat1), len(mat2), len(mat2[0])
    res = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            total = 0
            for t in range(k):
                total += mat1[i][t] * mat2[t][j]
            res[i][j] = total
    return res


for a, b, e in examples:
    assert multiply_brute(a, b) == e
print("PASS ví dụ (brute force)")

PASS ví dụ (brute force)


**Độ phức tạp (Cách 1).** Time `O(m·k·n)` (tối đa `10⁶` cho `m = k = n = 100`: vẫn chạy được, nhưng lãng phí khi thưa); Space `O(m·n)` cho kết quả.

**Nút thắt.** Mỗi phần tử của `mat1` bằng `0` vẫn bị nhân với cả một hàng `mat2`. Cần bỏ qua.

## 6. Ý tưởng thuật toán / bước đột phá

Hai bước:

1. **Đổi thứ tự vòng lặp** thành `i → k → j`: với mỗi `A[i][k]`, cộng `A[i][k] · B[k][j]` vào `C[i][j]` cho mọi `j`. Kết quả không đổi, nhưng điều kiện `A[i][k] == 0` giờ cho phép **bỏ cả vòng `j`**.
2. **Nén dữ liệu thưa:** chỉ lưu và duyệt phần tử khác 0 của `A` (và của `B`). Công việc tỷ lệ với số cặp `(A[i][k], B[k][j])` **cùng khác 0**, không phải `m·k·n`.

Đây là ý tưởng quen thuộc: **không làm việc thừa**, và **dùng cấu trúc dữ liệu hợp với dữ liệu** (danh sách kề thay vì mảng đặc).

In [8]:
# Đo lợi ích: đếm số phép nhân khi ma trận có xác suất khác 0 là density
def random_matrix(rows, cols, density, rnd):
    return [[rnd.randint(-100, 100) if rnd.random() < density else 0 for _ in range(cols)] for _ in range(rows)]

def count_pairs(A, B):
    """Số cặp (A[i][t] != 0, B[t][j] != 0): số phép nhân tối thiểu cần làm."""
    cnt = 0
    for i in range(len(A)):
        for t in range(len(B)):
            if A[i][t]:
                cnt += sum(1 for v in B[t] if v)
    return cnt

rnd = random.Random(311)
for density in (1.0, 0.5, 0.1, 0.02):
    A = random_matrix(100, 100, density, rnd)
    B = random_matrix(100, 100, density, rnd)
    print(f"mật độ {density:>4}: brute = {100 ** 3:>8} phép nhân | bỏ qua A = {count_ikj_skip(A, B):>8} | cả hai thưa = {count_pairs(A, B):>8}")

mật độ  1.0: brute =  1000000 phép nhân | bỏ qua A =   994300 | cả hai thưa =   987551
mật độ  0.5: brute =  1000000 phép nhân | bỏ qua A =   499900 | cả hai thưa =   253047
mật độ  0.1: brute =  1000000 phép nhân | bỏ qua A =    94500 | cả hai thưa =     8904
mật độ 0.02: brute =  1000000 phép nhân | bỏ qua A =    19800 | cả hai thưa =      367


## 7. Các cách tiếp theo

### Cách 2 — Thứ tự `i, k, j` và bỏ qua `A[i][k] == 0`

**Giải thích.** Với mỗi `i`, duyệt `k`; nếu `A[i][k] == 0` thì `continue`; ngược lại cộng `A[i][k] · B[k][j]` vào `res[i][j]` cho mọi `j`. Đây là cách phổ biến nhất ở các lời giải tham khảo.

In [9]:
def multiply_skip_zero(mat1: List[List[int]], mat2: List[List[int]]) -> List[List[int]]:
    m, k, n = len(mat1), len(mat2), len(mat2[0])
    res = [[0] * n for _ in range(m)]
    for i in range(m):
        for t in range(k):
            a = mat1[i][t]
            if a == 0:
                continue                      # bỏ qua cả vòng j
            row_b = mat2[t]
            row_r = res[i]
            for j in range(n):
                row_r[j] += a * row_b[j]
    return res


for a, b, e in examples:
    assert multiply_skip_zero(a, b) == e
print("PASS ví dụ (bỏ qua số 0 của A)")

PASS ví dụ (bỏ qua số 0 của A)


**Độ phức tạp Cách 2:** Time `O(m·k + nnz(A)·n)` (tối đa `O(m·k·n)`); Space `O(m·n)`.

### Cách 3 — Nén hàng của `A` thành danh sách `(k, giá trị)`

**Giải thích.** Trước tiên đổi mỗi hàng của `A` thành danh sách các cặp khác 0. Khi nhân, chỉ duyệt các cặp đó: không còn phải quét cả `k` cột của hàng. Phần `B` vẫn dạng đặc.

In [10]:
def multiply_compress_a(mat1: List[List[int]], mat2: List[List[int]]) -> List[List[int]]:
    m, n = len(mat1), len(mat2[0])
    rows_a = [[(t, v) for t, v in enumerate(row) if v != 0] for row in mat1]
    res = [[0] * n for _ in range(m)]
    for i in range(m):
        row_r = res[i]
        for t, a in rows_a[i]:
            row_b = mat2[t]
            for j in range(n):
                row_r[j] += a * row_b[j]
    return res


for a, b, e in examples:
    assert multiply_compress_a(a, b) == e
print("PASS ví dụ (nén A)")

PASS ví dụ (nén A)


**Độ phức tạp Cách 3:** Time `O(m·k + nnz(A)·n)`; vòng duyệt A giờ là `O(nnz(A))` sau bước nén `O(m·k)`.

### Cách 4 — Nén cả hai ma trận: chỉ nhân các cặp cùng khác 0

**Giải thích.** Nén hàng `t` của `B` thành `[(j, w)]` khác 0. Với mỗi `A[i][t] ≠ 0` và mỗi `(j, w)` của hàng `t` ở `B`: cộng `A[i][t] · w` vào `res[i][j]`. Mỗi phép nhân đều là phép nhân **có nghĩa** (hai thừa số cùng khác 0).

In [11]:
def multiply_compress_both(mat1: List[List[int]], mat2: List[List[int]]) -> List[List[int]]:
    m, n = len(mat1), len(mat2[0])
    rows_a = [[(t, v) for t, v in enumerate(row) if v != 0] for row in mat1]
    rows_b = [[(j, v) for j, v in enumerate(row) if v != 0] for row in mat2]
    res = [[0] * n for _ in range(m)]
    for i in range(m):
        row_r = res[i]
        for t, a in rows_a[i]:
            for j, b in rows_b[t]:
                row_r[j] += a * b
    return res


for a, b, e in examples:
    assert multiply_compress_both(a, b) == e
print("PASS ví dụ (nén cả hai)")

PASS ví dụ (nén cả hai)


**Độ phức tạp Cách 4:** Time `O(m·k + k·n + Σ nnz(A cột t) · nnz(B hàng t))` (tức chỉ các cặp cùng khác 0); Space `O(nnz(A) + nnz(B) + m·n)`.

### Cách 5 — Từ điển thưa (dict), dùng cho ma trận rất lớn

**Giải thích.** Lưu `A` dưới dạng `{(i, t): v}` và `B` theo hàng `{t: [(j, w), ...]}`. Kết quả cũng gom trong dict `{(i, j): tổng}` rồi ghi vào ma trận đặc ở cuối. Dạng này mạnh khi ma trận rất lớn và rất thưa (kích thước vượt ràng buộc của đề), vì không tốn bộ nhớ cho ô trống.

In [12]:
def multiply_dict(mat1: List[List[int]], mat2: List[List[int]]) -> List[List[int]]:
    m, n = len(mat1), len(mat2[0])
    a_items = {(i, t): v for i, row in enumerate(mat1) for t, v in enumerate(row) if v != 0}
    b_rows = {}
    for t, row in enumerate(mat2):
        entries = [(j, v) for j, v in enumerate(row) if v != 0]
        if entries:
            b_rows[t] = entries
    acc = {}
    for (i, t), a in a_items.items():
        for j, b in b_rows.get(t, ()):
            acc[(i, j)] = acc.get((i, j), 0) + a * b
    res = [[0] * n for _ in range(m)]
    for (i, j), v in acc.items():
        res[i][j] = v
    return res


for a, b, e in examples:
    assert multiply_dict(a, b) == e
print("PASS ví dụ (dict thưa)")

PASS ví dụ (dict thưa)


**Độ phức tạp Cách 5:** Time `O(số cặp khác 0 khớp)`; Space `O(nnz(A) + nnz(B) + nnz(C))` (ngoài ma trận đặc kết quả).

> Cách 2 là cách nộp bài (ngắn, đủ nhanh). Cách 4 là cách "thưa thật sự". Cách 1 làm oracle.

## 8. TRACE từng bước

Theo dõi Cách 4 trên ví dụ 1: in các cặp `(i, t, a)` và từng phép cộng `res[i][j] += a·b`.

In [13]:
def trace_multiply(mat1, mat2):
    m, n = len(mat1), len(mat2[0])
    rows_a = [[(t, v) for t, v in enumerate(row) if v != 0] for row in mat1]
    rows_b = [[(j, v) for j, v in enumerate(row) if v != 0] for row in mat2]
    print("rows_a =", rows_a)
    print("rows_b =", rows_b)
    res = [[0] * n for _ in range(m)]
    mults = 0
    for i in range(m):
        for t, a in rows_a[i]:
            for j, b in rows_b[t]:
                res[i][j] += a * b
                mults += 1
                print(f"  A[{i}][{t}]={a} * B[{t}][{j}]={b} -> res[{i}][{j}] = {res[i][j]}")
    print("kết quả:", res, "| số phép nhân thực hiện:", mults, "(brute force cần", m * len(mat2) * n, ")")
    return res


assert trace_multiply(*examples[0][:2]) == examples[0][2]

rows_a = [[(0, 1)], [(0, -1), (2, 3)]]
rows_b = [[(0, 7)], [], [(2, 1)]]
  A[0][0]=1 * B[0][0]=7 -> res[0][0] = 7
  A[1][0]=-1 * B[0][0]=7 -> res[1][0] = -7
  A[1][2]=3 * B[2][2]=1 -> res[1][2] = 3
kết quả: [[7, 0, 0], [-7, 0, 3]] | số phép nhân thực hiện: 3 (brute force cần 18 )


Bảng tóm tắt (ví dụ 1): `rows_a = [[(0,1)], [(0,-1),(2,3)]]`, `rows_b = [[(0,7)], [], [(2,1)]]`.

| `(i, t)` | `a` | hàng `t` của B | cộng vào |
|---|---|---|---|
| `(0, 0)` | 1 | `[(0, 7)]` | `res[0][0] += 7` |
| `(1, 0)` | −1 | `[(0, 7)]` | `res[1][0] += −7` |
| `(1, 2)` | 3 | `[(2, 1)]` | `res[1][2] += 3` |

Chỉ `3` phép nhân thay vì `2·3·3 = 18`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (bỏ qua `A[i][k] == 0`), đúng chữ ký của LeetCode.

In [14]:
class Solution:
    def multiply(self, mat1: List[List[int]], mat2: List[List[int]]) -> List[List[int]]:
        m, k, n = len(mat1), len(mat2), len(mat2[0])
        ans = [[0] * n for _ in range(m)]
        for i in range(m):
            for x in range(k):
                if mat1[i][x] == 0:
                    continue
                for j in range(n):
                    ans[i][j] += mat1[i][x] * mat2[x][j]
        return ans


for a, b, e in examples:
    assert Solution().multiply(a, b) == e
print("PASS ví dụ đề cho")

PASS ví dụ đề cho


## 10. Vì sao thuật toán đúng

- **Đổi thứ tự vòng lặp không đổi kết quả:** `C[i][j] = Σₖ A[i][k]·B[k][j]` là tổng hữu hạn nên cộng theo thứ tự `k` ngoài `j` hay `j` ngoài `k` đều ra cùng tổng (phép cộng số nguyên giao hoán, kết hợp).
- **Bỏ qua `A[i][k] = 0`:** số hạng `0 · B[k][j] = 0` không đóng góp vào tổng, nên bỏ đi không đổi kết quả.
- **Nén `B`:** tương tự, các cặp có `B[k][j] = 0` đóng góp `0`.
- **Kết quả đúng chiều:** `res` có `m` hàng (số hàng của `mat1`), `n` cột (số cột của `mat2`).
- **Mỗi `res[i][j]` khởi tạo `0` và chỉ được cộng**: các số hạng `k` khác nhau được gộp bằng `+=`, không ghi đè.

## 11. Phân tích Time / Space Complexity

Gọi `m, k, n` như đề, `nnz(·)` là số phần tử khác 0.

- **Cách 1:** Time `O(m·k·n)`; Space `O(m·n)`.
- **Cách 2:** Time `O(m·k + nnz(A)·n)`; Space `O(m·n)`.
- **Cách 3:** Time `O(m·k + nnz(A)·n)`; Space `O(nnz(A) + m·n)`.
- **Cách 4:** Time `O(m·k + k·n + P)` với `P` là số cặp `(A[i][t], B[t][j])` cùng khác 0; Space `O(nnz(A) + nnz(B) + m·n)`.
- **Cách 5:** Time `O(P)`; Space `O(nnz(A) + nnz(B) + nnz(C))`.
- Trường hợp xấu nhất (ma trận đặc) mọi cách đều `O(m·k·n)` = `10⁶` với ràng buộc của đề.

## 12. Bộ test

Mô hình độc lập: nhân qua **ma trận chuyển vị** và `zip` (`oracle_zip`) không dùng chỉ số, đối chiếu với brute force ba vòng lặp. Mọi cách nhanh được so với `multiply_brute`.

In [15]:
def oracle_zip(A, B):
    cols_b = list(zip(*B))                               # cột của B
    return [[sum(x * y for x, y in zip(row, col)) for col in cols_b] for row in A]


impls = {
    "brute": multiply_brute,
    "skip_zero": multiply_skip_zero,
    "compress_a": multiply_compress_a,
    "compress_both": multiply_compress_both,
    "dict": multiply_dict,
    "submit": lambda a, b: Solution().multiply(a, b),
}


def check(A, B, expected=None):
    base = multiply_brute(A, B)
    assert base == oracle_zip(A, B)
    if expected is not None:
        assert base == expected, (A, B, base, expected)
    for name, fn in impls.items():
        assert fn([r[:] for r in A], [r[:] for r in B]) == base, (name, A, B)
    assert len(base) == len(A) and all(len(r) == len(B[0]) for r in base)
    return base


for a, b, e in examples:
    check(a, b, e)
print("PASS 2 ví dụ của đề")

PASS 2 ví dụ của đề


### 12.1. Các trường hợp biên và giá trị tính tay

In [16]:
cases = [
    ([[5]], [[3]], [[15]], "1x1"),
    ([[0]], [[7]], [[0]], "một số 0"),
    ([[1, 2, 3]], [[4], [5], [6]], [[32]], "hàng x cột = số"),
    ([[1], [2], [3]], [[4, 5, 6]], [[4, 5, 6], [8, 10, 12], [12, 15, 18]], "cột x hàng = ma trận"),
    ([[1, 0], [0, 1]], [[9, 8], [7, 6]], [[9, 8], [7, 6]], "nhân ma trận đơn vị"),
    ([[0, 0], [0, 0]], [[1, 2], [3, 4]], [[0, 0], [0, 0]], "ma trận không"),
    ([[1, 2], [3, 4]], [[5, 6], [7, 8]], [[19, 22], [43, 50]], "ma trận đặc 2x2"),
    ([[-100, 100]], [[100], [-100]], [[-20000]], "giá trị biên +-100"),
    ([[1, -1], [-1, 1]], [[1, -1], [-1, 1]], [[2, -2], [-2, 2]], "có số âm"),
    ([[2, 0, 0], [0, 0, 0]], [[0, 5], [0, 0], [0, 0]], [[0, 10], [0, 0]], "cỡ 2x3 nhân 3x2"),
]
for a, b, e, why in cases:
    assert check(a, b, e) == e
    print("PASS", why)

PASS 1x1
PASS một số 0
PASS hàng x cột = số
PASS cột x hàng = ma trận
PASS nhân ma trận đơn vị
PASS ma trận không
PASS ma trận đặc 2x2
PASS giá trị biên +-100
PASS có số âm
PASS cỡ 2x3 nhân 3x2


### 12.2. Tính chất và phản ví dụ

In [17]:
rnd = random.Random(311)


def transpose(M):
    return [list(r) for r in zip(*M)]


for _ in range(200):
    m, k, n = rnd.randint(1, 6), rnd.randint(1, 6), rnd.randint(1, 6)
    d = rnd.choice([0.2, 0.6, 1.0])
    A = random_matrix(m, k, d, rnd)
    B = random_matrix(k, n, d, rnd)
    C = random_matrix(n, rnd.randint(1, 5), d, rnd)
    AB = multiply_compress_both(A, B)
    # (AB)^T = B^T A^T
    assert transpose(AB) == multiply_compress_both(transpose(B), transpose(A))
    # kết hợp: (AB)C = A(BC)
    assert multiply_compress_both(AB, C) == multiply_compress_both(A, multiply_compress_both(B, C))
    # nhân đơn vị
    I = [[1 if i == j else 0 for j in range(k)] for i in range(k)]
    assert multiply_compress_both(A, I) == A
    # tuyến tính: (2A)B = 2(AB)
    assert multiply_compress_both([[2 * x for x in r] for r in A], B) == [[2 * x for x in r] for r in AB]
print("PASS tính chất: (AB)^T = B^T A^T, kết hợp, nhân đơn vị, tuyến tính")

# Phản ví dụ: nhân ma trận KHÔNG giao hoán
A, B = [[1, 2], [3, 4]], [[0, 1], [1, 0]]
assert multiply_brute(A, B) != multiply_brute(B, A)
print("PASS phản ví dụ: AB != BA:", multiply_brute(A, B), multiply_brute(B, A))

PASS tính chất: (AB)^T = B^T A^T, kết hợp, nhân đơn vị, tuyến tính
PASS phản ví dụ: AB != BA: [[2, 1], [4, 3]] [[3, 4], [1, 2]]


### 12.3. Đối chiếu ngẫu nhiên (có seed) với brute force

In [18]:
rnd = random.Random(2026)
for trial in range(800):
    m, k, n = rnd.randint(1, 8), rnd.randint(1, 8), rnd.randint(1, 8)
    d = rnd.choice([0.0, 0.1, 0.3, 0.7, 1.0])
    check(random_matrix(m, k, d, rnd), random_matrix(k, n, d, rnd))

for trial in range(12):                                   # cỡ lớn theo đề: 100 x 100
    d = rnd.choice([0.02, 0.1, 0.5, 1.0])
    check(random_matrix(100, 100, d, rnd), random_matrix(100, 100, d, rnd))

for shape in [(100, 1, 100), (1, 100, 1), (1, 1, 100), (100, 100, 1)]:     # hình dạng dị biệt
    m, k, n = shape
    check(random_matrix(m, k, 0.5, rnd), random_matrix(k, n, 0.5, rnd))
print("PASS 800 test nhỏ, 12 test 100x100, 4 hình dạng dị biệt")

PASS 800 test nhỏ, 12 test 100x100, 4 hình dạng dị biệt


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tạo kết quả bằng `[[0] * n] * m` (các hàng dùng chung)

In [19]:
def sai_hang_dung_chung(mat1, mat2):
    m, k, n = len(mat1), len(mat2), len(mat2[0])
    res = [[0] * n] * m                                 # sai: m tham chiếu tới CÙNG một hàng
    for i in range(m):
        for t in range(k):
            if mat1[i][t] == 0:
                continue
            for j in range(n):
                res[i][j] += mat1[i][t] * mat2[t][j]
    return res


a, b, e = examples[0]
print("hàng dùng chung ->", sai_hang_dung_chung(a, b), "| đúng:", e)
assert sai_hang_dung_chung(a, b) != e

hàng dùng chung -> [[0, 0, 3], [0, 0, 3]] | đúng: [[7, 0, 0], [-7, 0, 3]]


**Vì sao sai:** `[[0] * n] * m` sao chép **tham chiếu** nên mọi hàng là cùng một đối tượng; cộng vào hàng `i` cũng làm đổi mọi hàng khác. Luôn dùng `[[0] * n for _ in range(m)]`.

### Lỗi 2: Dùng `=` thay vì `+=`

In [20]:
def sai_ghi_de(mat1, mat2):
    m, k, n = len(mat1), len(mat2), len(mat2[0])
    res = [[0] * n for _ in range(m)]
    for i in range(m):
        for t in range(k):
            for j in range(n):
                res[i][j] = mat1[i][t] * mat2[t][j]       # sai: ghi đè thay vì cộng dồn
    return res


print("ghi đè ->", sai_ghi_de([[1, 2]], [[3], [4]]), "| đúng: [[11]]")
assert sai_ghi_de([[1, 2]], [[3], [4]]) != [[11]]

ghi đè -> [[8]] | đúng: [[11]]


**Vì sao sai:** mỗi `C[i][j]` là **tổng** của `k` số hạng; ghi đè giữ lại số hạng cuối cùng (`2·4 = 8`) thay vì `1·3 + 2·4 = 11`.

### Lỗi 3: Dùng `break` thay vì `continue` khi gặp số 0

In [21]:
def sai_break(mat1, mat2):
    m, k, n = len(mat1), len(mat2), len(mat2[0])
    res = [[0] * n for _ in range(m)]
    for i in range(m):
        for t in range(k):
            if mat1[i][t] == 0:
                break                                    # sai: bỏ cả phần còn lại của hàng
            for j in range(n):
                res[i][j] += mat1[i][t] * mat2[t][j]
    return res


print("break ->", sai_break([[0, 1]], [[5], [6]]), "| đúng: [[6]]")
assert sai_break([[0, 1]], [[5], [6]]) != [[6]]

break -> [[0]] | đúng: [[6]]


**Vì sao sai:** số `0` chỉ nên bỏ qua **chính nó** (`continue`), không phải kết thúc cả hàng. Phần tử khác 0 phía sau vẫn đóng góp.

### Lỗi 4: Nhầm chiều của ma trận kết quả

In [22]:
def sai_chieu(mat1, mat2):
    m, k, n = len(mat1), len(mat2), len(mat2[0])
    res = [[0] * m for _ in range(n)]                    # sai: phải là m hàng, n cột
    for i in range(m):
        for t in range(k):
            for j in range(n):
                res[i][j] += mat1[i][t] * mat2[t][j]
    return res


try:
    out = sai_chieu([[1, 2, 3], [4, 5, 6]], [[1], [1], [1]])      # đúng phải là [[6], [15]]
    print("nhầm chiều ->", out)
except IndexError as exc:
    print("nhầm chiều -> IndexError:", exc, "| đúng: [[6], [15]]")

nhầm chiều -> IndexError: list index out of range | đúng: [[6], [15]]


**Vì sao sai:** kết quả của `m × k` nhân `k × n` có `m` hàng và `n` cột. Nhầm sẽ lỗi `IndexError` khi `m != n`, hoặc cho kết quả sai hình dạng khi `m == n` nhưng nhầm ở nơi khác. Hãy viết sẵn comment kích thước trước khi code.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Ma trận thưa" trong đề = **hãy bỏ qua các số 0**; đây là gợi ý tối ưu chính.
- Thứ tự `i → k → j` cho phép `continue` sớm ở vòng giữa; ngoài ra truy cập `mat2[k]` theo hàng thân thiện với bộ nhớ đệm.
- Nén sang danh sách `(chỉ số, giá trị)` hoặc dict khi ma trận rất lớn và rất thưa.
- Kết quả: `[[0] * n for _ in range(m)]`, **không** dùng `[[0] * n] * m`.
- Để kiểm tra chéo: `oracle_zip` dùng `zip(*B)` (cột của `B`).
- Dùng tính chất toán: `(AB)ᵀ = BᵀAᵀ`, nhân đơn vị, kết hợp để test nhanh.

## 15. Biến thể của bài

1. **Nhân ma trận với vector** (`y = A·x`).
2. **Lũy thừa ma trận vuông** `Aᵏ` (bình phương liên tiếp).
3. **Lưu ma trận dạng CSR** (Compressed Sparse Row).
4. **Đếm số phần tử khác 0 của kết quả.**
5. **Nhân ma trận theo modulo** `10⁹ + 7`.

In [23]:
def mat_vec(A, x):
    """Biến thể 1: A (m x k) nhân vector x (k)."""
    return [sum(v * x[t] for t, v in enumerate(row) if v) for row in A]


def mat_pow(A, p):
    """Biến thể 2: A^p cho ma trận vuông, p >= 0, bình phương liên tiếp dùng multiply_compress_both."""
    n = len(A)
    result = [[1 if i == j else 0 for j in range(n)] for i in range(n)]
    base = [row[:] for row in A]
    while p:
        if p & 1:
            result = multiply_compress_both(result, base)
        base = multiply_compress_both(base, base)
        p >>= 1
    return result


def to_csr(A):
    """Biến thể 3: CSR gồm (values, col_idx, row_ptr)."""
    values, col_idx, row_ptr = [], [], [0]
    for row in A:
        for j, v in enumerate(row):
            if v:
                values.append(v)
                col_idx.append(j)
        row_ptr.append(len(values))
    return values, col_idx, row_ptr


def csr_multiply_vec(csr, x):
    values, col_idx, row_ptr = csr
    return [sum(values[p] * x[col_idx[p]] for p in range(row_ptr[i], row_ptr[i + 1])) for i in range(len(row_ptr) - 1)]


def count_nonzero_result(A, B):
    """Biến thể 4."""
    return sum(1 for row in multiply_dict(A, B) for v in row if v)


def multiply_mod(A, B, mod=10 ** 9 + 7):
    """Biến thể 5: nhân modulo."""
    m, n = len(A), len(B[0])
    res = [[0] * n for _ in range(m)]
    for i in range(m):
        for t, a in enumerate(A[i]):
            if a:
                for j in range(n):
                    res[i][j] = (res[i][j] + a * B[t][j]) % mod
    return res


rnd = random.Random(15)
for _ in range(100):
    n = rnd.randint(1, 6)
    A = random_matrix(n, n, 0.5, rnd)
    x = [rnd.randint(-5, 5) for _ in range(n)]
    assert mat_vec(A, x) == [r[0] for r in multiply_brute(A, [[v] for v in x])]
    assert csr_multiply_vec(to_csr(A), x) == mat_vec(A, x)
    p = rnd.randint(0, 6)
    expect = [[1 if i == j else 0 for j in range(n)] for i in range(n)]
    for _ in range(p):
        expect = multiply_brute(expect, A)
    assert mat_pow(A, p) == expect
    assert count_nonzero_result(A, A) == sum(1 for r in multiply_brute(A, A) for v in r if v)
    mod = 10 ** 9 + 7
    assert multiply_mod(A, A, mod) == [[v % mod for v in r] for r in multiply_brute(A, A)]
assert mat_pow([[1, 1], [1, 0]], 10) == [[89, 55], [55, 34]]        # Fibonacci
print("PASS biến thể: ma trận x vector, lũy thừa (Fibonacci), CSR, đếm khác 0, modulo")

PASS biến thể: ma trận x vector, lũy thừa (Fibonacci), CSR, đếm khác 0, modulo


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ba vòng lặp | theo định nghĩa | `O(m·k·n)` | `O(m·n)` | oracle |
| 2. `i, k, j` + `continue` | bỏ qua `A[i][k] = 0` | `O(m·k + nnz(A)·n)` | `O(m·n)` | **cách nộp bài** |
| 3. Nén hàng `A` | duyệt cặp khác 0 của `A` | `O(m·k + nnz(A)·n)` | `O(nnz(A) + m·n)` | `A` rất thưa |
| 4. Nén cả hai | chỉ nhân cặp cùng khác 0 | `O(P)` + nén | `O(nnz + m·n)` | cả hai thưa |
| 5. dict thưa | `{(i, j): tổng}` | `O(P)` | `O(nnz(A) + nnz(B) + nnz(C))` | ma trận cực lớn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `multiply_brute([[1, 2], [3, 4]], [[1, 0], [0, 1]])` và `multiply_brute([[2]], [[0]])`.

**Bài 2 (tính tay).** Tính `[[1, 0, 2], [0, 3, 0]] × [[1, 0], [0, 1], [4, 0]]`.

**Bài 3 (điền TODO).** Hoàn thành `multiply_todo` theo Cách 2.

In [24]:
def multiply_todo(mat1, mat2):
    # TODO 1: lấy m = len(mat1), k = len(mat2), n = len(mat2[0])
    # TODO 2: res = [[0] * n for _ in range(m)]
    # TODO 3: for i, for t: nếu mat1[i][t] == 0 thì continue
    # TODO 4: for j: res[i][j] += mat1[i][t] * mat2[t][j]
    # TODO 5: trả res
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `multiply_csr(A, B)`: đổi `A` sang CSR (`to_csr`) rồi nhân với `B` đặc, dùng `values`, `col_idx`, `row_ptr`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `multiply_chain(mats)` nhân một dãy ma trận. (b) Viết `diag_of_product(A, B)` chỉ trả về **đường chéo** của `A·B` (ma trận vuông) trong `O(m·k)` mà không tính cả `A·B`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: nhân với ma trận đơn vị không đổi gì; nhân với `0` ra `0`. Bài 2: tính từng `C[i][j]` bằng tích vô hướng hàng-cột.
- **Mức 2 (vừa):** Bài 1: `[[1, 2], [3, 4]]` và `[[0]]`. Bài 2: kết quả `[[9, 0], [0, 3]]` (hàng 0: `1·1 + 0·0 + 2·4 = 9`, `1·0 + 0·1 + 2·0 = 0`; hàng 1: `3` ở vị trí `[1][1]`). Bài 3: năm TODO trong comment.
- **Mức 3 (sâu):** Bài 4: với mỗi `i`, duyệt `p` từ `row_ptr[i]` đến `row_ptr[i + 1]`, `t = col_idx[p]`, `a = values[p]`, rồi cộng `a · B[t]` vào hàng `i`. Bài 5(a): dùng `functools.reduce`. Bài 5(b): `diag[i] = Σₜ A[i][t]·B[t][i]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [25]:
from functools import reduce

# Bài 1
assert multiply_brute([[1, 2], [3, 4]], [[1, 0], [0, 1]]) == [[1, 2], [3, 4]]
assert multiply_brute([[2]], [[0]]) == [[0]]

# Bài 2
assert multiply_brute([[1, 0, 2], [0, 3, 0]], [[1, 0], [0, 1], [4, 0]]) == [[9, 0], [0, 3]]

# Bài 3
def multiply_todo_ans(mat1, mat2):
    m, k, n = len(mat1), len(mat2), len(mat2[0])
    res = [[0] * n for _ in range(m)]
    for i in range(m):
        for t in range(k):
            if mat1[i][t] == 0:
                continue
            for j in range(n):
                res[i][j] += mat1[i][t] * mat2[t][j]
    return res


# Bài 4
def multiply_csr(A, B):
    values, col_idx, row_ptr = to_csr(A)
    n = len(B[0])
    res = [[0] * n for _ in range(len(A))]
    for i in range(len(A)):
        for p in range(row_ptr[i], row_ptr[i + 1]):
            t, a = col_idx[p], values[p]
            for j in range(n):
                res[i][j] += a * B[t][j]
    return res


# Bài 5
def multiply_chain(mats):
    return reduce(multiply_compress_both, mats)


def diag_of_product(A, B):
    return [sum(A[i][t] * B[t][i] for t in range(len(B))) for i in range(len(A))]


rnd = random.Random(19)
for _ in range(200):
    m, k, n = rnd.randint(1, 6), rnd.randint(1, 6), rnd.randint(1, 6)
    A = random_matrix(m, k, 0.4, rnd)
    B = random_matrix(k, n, 0.4, rnd)
    C = random_matrix(n, rnd.randint(1, 5), 0.4, rnd)
    base = multiply_brute(A, B)
    assert multiply_todo_ans(A, B) == base
    assert multiply_csr(A, B) == base
    assert multiply_chain([A, B, C]) == multiply_brute(base, C)
    S = random_matrix(m, m, 0.4, rnd)
    T = random_matrix(m, m, 0.4, rnd)
    full = multiply_brute(S, T)
    assert diag_of_product(S, T) == [full[i][i] for i in range(m)]
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (thứ tự `i, k, j`, bỏ qua số 0)

```python
class Solution:
    def multiply(self, mat1: List[List[int]], mat2: List[List[int]]) -> List[List[int]]:
        m, k, n = len(mat1), len(mat2), len(mat2[0])
        ans = [[0] * n for _ in range(m)]
        for i in range(m):
            for x in range(k):
                if mat1[i][x] == 0:
                    continue
                for j in range(n):
                    ans[i][j] += mat1[i][x] * mat2[x][j]
        return ans
```

**Checklist**

- [ ] Kích thước kết quả là `m × n` (`m` hàng của `mat1`, `n` cột của `mat2`)?
- [ ] Tạo kết quả bằng `[[0] * n for _ in range(m)]` (không dùng `* m`)?
- [ ] Cộng dồn bằng `+=`, không ghi đè?
- [ ] Bỏ qua số 0 bằng `continue`, không `break`?
- [ ] Vòng lặp theo thứ tự `i, k, j` để loại cả vòng `j`?
- [ ] Đã thử: `1×1`, toàn số 0, ma trận đơn vị, `m != n`, giá trị biên `±100`?
- [ ] Đã đối chiếu với brute force / oracle `zip(*B)` trên input ngẫu nhiên có seed?
- [ ] Skeleton đúng `class Solution` / `multiply(self, mat1, mat2)`?